[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/algoritmos-UdeA/logica1-20262/blob/main/clases/14/teoria/notebooks/notas_finales_paso_a_paso.ipynb) [![Built with AI](https://img.shields.io/badge/Built%20with-AI-blue.svg)](https://shields.io/)

# Notas finales de un curso con una matriz, paso a paso

Este notebook construye, función por función, el programa integrador de la [clase 14](https://github.com/algoritmos-UdeA/logica1-20262/blob/main/clases/14/README.md): las notas de un curso en una **matriz**, los nombres en un arreglo paralelo a las **filas** y los pesos de los parciales en un arreglo paralelo a las **columnas**.

Cada función se prueba con `assert` **antes** de pasar a la siguiente. El código es el mismo de la carpeta [`notas_matriz/`](https://github.com/algoritmos-UdeA/logica1-20262/tree/main/clases/14/teoria/codigo/notas_matriz).

## Cómo usar este notebook

1. Se puede abrir en Google Colab con el botón **Open in Colab** de arriba, sin instalar nada, o en Jupyter (Anaconda Navigator o la extensión de Jupyter de VS Code).
2. Las celdas se ejecutan **en orden**, de arriba hacia abajo, con **Shift + Enter**: casi todas usan lo definido en celdas anteriores.
3. Las salidas ya están guardadas, pero conviene **predecir** qué va a imprimir cada celda antes de ejecutarla.
4. Algunas celdas **fallan a propósito** (están marcadas con ⚠️). Por eso no conviene usar *ejecutar todo*: la ejecución se detiene en el primer error.
5. Si algo se comporta de forma extraña, se reinicia y se vuelve a ejecutar desde el principio: en Colab, **Entorno de ejecución → Reiniciar sesión**; en Jupyter, **Kernel → Restart Kernel**.

| Marca | Significado |
|---|---|
| 🔮 **Prediga** | Antes de ejecutar la celda, responda en papel qué va a pasar |
| ⚠️ **Error frecuente** | La celda siguiente falla o da un resultado equivocado **a propósito** |
| ✏️ **Hágalo usted mismo** | Un ejercicio; la solución está oculta y se despliega con un clic |
| 🔎 **Python Tutor** | Debajo de las celdas que ejecutan algo: abre ese código en [Python Tutor](https://pythontutor.com/) para avanzar instrucción por instrucción. El enlace incluye, de las celdas anteriores, solo lo que la celda necesita y sin los comentarios de documentación |

---

## Parte 0 — Lo que se necesita saber

| Concepto | Dónde se usa aquí |
|---|---|
| Matriz: lista de listas, `notas[i][j]` | Las notas: una fila por estudiante, una columna por parcial |
| Acumular por fila (el acumulador se reinicia en cada fila) | La nota final de cada estudiante |
| Buscar el mayor y el menor de un arreglo | La nota final más alta y la más baja |
| Contar, crear del tamaño justo y llenar (sin `append`) | Los estudiantes empatados |
| Funciones con *Supone / Garantiza* y pruebas con `assert` | Todas las funciones |

---

## Parte 1 — El problema y un caso calculado a mano

> Se tienen los nombres de los estudiantes de un curso y sus notas en tres parciales, que valen 30 %, 30 % y 40 %. Calcular la nota final de cada estudiante y mostrar quiénes obtuvieron la nota final más alta y quiénes la más baja.

| Estudiante | P1 (30 %) | P2 (30 %) | P3 (40 %) | Nota final |
|---|---|---|---|---|
| Ana | 3.0 | 3.5 | 2.5 | 0.9 + 1.05 + 1.0 = **2.95** |
| Beto | 4.5 | 4.0 | 5.0 | 1.35 + 1.2 + 2.0 = **4.55** (la más alta) |
| Carla | 2.0 | 3.0 | 3.5 | 0.6 + 0.9 + 1.4 = **2.90** (la más baja) |
| Dario | 4.0 | 4.5 | 3.0 | 1.2 + 1.35 + 1.2 = **3.75** |

Este caso calculado a mano es el que deben dar las pruebas.

---

## Parte 2 — El diseño

### Los datos

`nombres` es paralelo a las **filas** de `notas`, y `PESOS` a sus **columnas**. El índice `j` sirve para entrar a la matriz y al arreglo de pesos.

In [ ]:
nombres = ["Ana", "Beto", "Carla", "Dario"]
notas = [[3.0, 3.5, 2.5],
         [4.5, 4.0, 5.0],
         [2.0, 3.0, 3.5],
         [4.0, 4.5, 3.0]]
PESOS = [0.3, 0.3, 0.4]

print(nombres[1], notas[1])
print("Peso del parcial 3:", PESOS[2])

Beto [4.5, 4.0, 5.0]
Peso del parcial 3: 0.4


🔎 [Ver paso a paso en Python Tutor](https://pythontutor.com/visualize.html#code=nombres%20%3D%20%5B%22Ana%22%2C%20%22Beto%22%2C%20%22Carla%22%2C%20%22Dario%22%5D%0Anotas%20%3D%20%5B%5B3.0%2C%203.5%2C%202.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.5%2C%204.0%2C%205.0%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B2.0%2C%203.0%2C%203.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.0%2C%204.5%2C%203.0%5D%5D%0APESOS%20%3D%20%5B0.3%2C%200.3%2C%200.4%5D%0A%0Aprint%28nombres%5B1%5D%2C%20notas%5B1%5D%29%0Aprint%28%22Peso%20del%20parcial%203%3A%22%2C%20PESOS%5B2%5D%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=311&rawInputLstJSON=%5B%5D&textReferences=false)

### Las funciones (diseño top-down)

![Árbol top-down del programa](https://raw.githubusercontent.com/algoritmos-UdeA/logica1-20262/main/clases/14/teoria/images/arbol_notas_matriz.png)

Se construyen de abajo hacia arriba en tres pasos: **calcular**, **buscar** y **mostrar a los empatados**.

---

## Parte 3 — Paso 1: `calcular_notas_finales`

Recibe una matriz y un arreglo, y retorna un **arreglo nuevo**. Es un acumulador **por fila**, pero cada nota se multiplica por el peso de **su columna**.

In [ ]:
def calcular_notas_finales(notas, pesos):
    """
    Supone:
        notas es una matriz de numeros con al menos una fila, y pesos
        es una lista con un peso por columna de notas; los pesos suman
        1.0.
    Garantiza:
        Retorna una lista nueva con la nota final de cada fila: la suma
        de notas[i][j] * pesos[j]. No modifica notas ni pesos.
    """
    n = len(notas)
    finales = [0.0] * n
    for i in range(n):
        suma = 0.0
        for j in range(len(pesos)):
            suma = suma + notas[i][j] * pesos[j]
        finales[i] = suma
    return finales

🔍 **Para entender.** La misma cuenta para la fila de Ana, mostrando cada vuelta:

In [ ]:
suma = 0.0
for j in range(len(PESOS)):
    suma = suma + notas[0][j] * PESOS[j]
    print(f"j = {j}: notas[0][{j}] = {notas[0][j]}, peso {PESOS[j]}, suma = {suma}")

j = 0: notas[0][0] = 3.0, peso 0.3, suma = 0.8999999999999999
j = 1: notas[0][1] = 3.5, peso 0.3, suma = 1.95
j = 2: notas[0][2] = 2.5, peso 0.4, suma = 2.95


🔎 [Ver paso a paso en Python Tutor](https://pythontutor.com/visualize.html#code=nombres%20%3D%20%5B%22Ana%22%2C%20%22Beto%22%2C%20%22Carla%22%2C%20%22Dario%22%5D%0Anotas%20%3D%20%5B%5B3.0%2C%203.5%2C%202.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.5%2C%204.0%2C%205.0%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B2.0%2C%203.0%2C%203.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.0%2C%204.5%2C%203.0%5D%5D%0APESOS%20%3D%20%5B0.3%2C%200.3%2C%200.4%5D%0A%0Asuma%20%3D%200.0%0Afor%20j%20in%20range%28len%28PESOS%29%29%3A%0A%20%20%20%20suma%20%3D%20suma%20%2B%20notas%5B0%5D%5Bj%5D%20%2A%20PESOS%5Bj%5D%0A%20%20%20%20print%28f%22j%20%3D%20%7Bj%7D%3A%20notas%5B0%5D%5B%7Bj%7D%5D%20%3D%20%7Bnotas%5B0%5D%5Bj%5D%7D%2C%20peso%20%7BPESOS%5Bj%5D%7D%2C%20suma%20%3D%20%7Bsuma%7D%22%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=311&rawInputLstJSON=%5B%5D&textReferences=false)

El primer paso no da 0.9 sino `0.8999999999999999`: los números reales se guardan con un pequeño error. En la celda siguiente, la nota de Carla sale `2.9000000000000004`. Por eso las pruebas comparan con `round(..., 2)` y la salida usa `:.2f`.

In [ ]:
finales = calcular_notas_finales(notas, PESOS)
print(finales)

assert len(finales) == 4
assert round(finales[0], 2) == 2.95                                  # tipico
assert round(finales[1], 2) == 4.55
assert calcular_notas_finales([[4.0, 2.0]], [0.5, 0.5]) == [3.0]     # limite: una fila
assert calcular_notas_finales([[3.0, 5.0]], [1.0, 0.0]) == [3.0]     # especial: peso 0
print("Paso 1: todas las pruebas pasaron")

[2.95, 4.55, 2.9000000000000004, 3.75]
Paso 1: todas las pruebas pasaron


🔎 [Ver paso a paso en Python Tutor](https://pythontutor.com/visualize.html#code=def%20calcular_notas_finales%28notas%2C%20pesos%29%3A%0A%20%20%20%20n%20%3D%20len%28notas%29%0A%20%20%20%20finales%20%3D%20%5B0.0%5D%20%2A%20n%0A%20%20%20%20for%20i%20in%20range%28n%29%3A%0A%20%20%20%20%20%20%20%20suma%20%3D%200.0%0A%20%20%20%20%20%20%20%20for%20j%20in%20range%28len%28pesos%29%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20suma%20%3D%20suma%20%2B%20notas%5Bi%5D%5Bj%5D%20%2A%20pesos%5Bj%5D%0A%20%20%20%20%20%20%20%20finales%5Bi%5D%20%3D%20suma%0A%20%20%20%20return%20finales%0A%0Anombres%20%3D%20%5B%22Ana%22%2C%20%22Beto%22%2C%20%22Carla%22%2C%20%22Dario%22%5D%0Anotas%20%3D%20%5B%5B3.0%2C%203.5%2C%202.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.5%2C%204.0%2C%205.0%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B2.0%2C%203.0%2C%203.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.0%2C%204.5%2C%203.0%5D%5D%0APESOS%20%3D%20%5B0.3%2C%200.3%2C%200.4%5D%0A%0Afinales%20%3D%20calcular_notas_finales%28notas%2C%20PESOS%29%0Aprint%28finales%29%0A%0Aassert%20len%28finales%29%20%3D%3D%204%0Aassert%20round%28finales%5B0%5D%2C%202%29%20%3D%3D%202.95%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%20tipico%0Aassert%20round%28finales%5B1%5D%2C%202%29%20%3D%3D%204.55%0Aassert%20calcular_notas_finales%28%5B%5B4.0%2C%202.0%5D%5D%2C%20%5B0.5%2C%200.5%5D%29%20%3D%3D%20%5B3.0%5D%20%20%20%20%20%23%20limite%3A%20una%20fila%0Aassert%20calcular_notas_finales%28%5B%5B3.0%2C%205.0%5D%5D%2C%20%5B1.0%2C%200.0%5D%29%20%3D%3D%20%5B3.0%5D%20%20%20%20%20%23%20especial%3A%20peso%200%0Aprint%28%22Paso%201%3A%20todas%20las%20pruebas%20pasaron%22%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=311&rawInputLstJSON=%5B%5D&textReferences=false)

---

## Parte 4 — Paso 2: `buscar_mayor` y `buscar_menor`

Son las búsquedas del mayor y del menor de la clase 12, sobre el arreglo de notas finales.

In [ ]:
def buscar_mayor(valores):
    """
    Supone:
        valores es una lista de numeros con al menos un elemento.
    Garantiza:
        Retorna el mayor valor de la lista. No modifica valores.
    """
    mayor = valores[0]
    for i in range(1, len(valores)):
        if valores[i] > mayor:
            mayor = valores[i]
    return mayor


def buscar_menor(valores):
    """
    Supone:
        valores es una lista de numeros con al menos un elemento.
    Garantiza:
        Retorna el menor valor de la lista. No modifica valores.
    """
    menor = valores[0]
    for i in range(1, len(valores)):
        if valores[i] < menor:
            menor = valores[i]
    return menor

In [ ]:
finales = calcular_notas_finales(notas, PESOS)
print(f"Mas alta: {buscar_mayor(finales):.2f}")
print(f"Mas baja: {buscar_menor(finales):.2f}")

assert buscar_mayor([2.95, 4.55, 2.9, 3.75]) == 4.55
assert buscar_menor([2.95, 4.55, 2.9, 3.75]) == 2.9
assert buscar_mayor([3.5]) == 3.5                       # limite: un elemento
print("Paso 2: todas las pruebas pasaron")

Mas alta: 4.55
Mas baja: 2.90
Paso 2: todas las pruebas pasaron


🔎 [Ver paso a paso en Python Tutor](https://pythontutor.com/visualize.html#code=def%20calcular_notas_finales%28notas%2C%20pesos%29%3A%0A%20%20%20%20n%20%3D%20len%28notas%29%0A%20%20%20%20finales%20%3D%20%5B0.0%5D%20%2A%20n%0A%20%20%20%20for%20i%20in%20range%28n%29%3A%0A%20%20%20%20%20%20%20%20suma%20%3D%200.0%0A%20%20%20%20%20%20%20%20for%20j%20in%20range%28len%28pesos%29%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20suma%20%3D%20suma%20%2B%20notas%5Bi%5D%5Bj%5D%20%2A%20pesos%5Bj%5D%0A%20%20%20%20%20%20%20%20finales%5Bi%5D%20%3D%20suma%0A%20%20%20%20return%20finales%0A%0Adef%20buscar_mayor%28valores%29%3A%0A%20%20%20%20mayor%20%3D%20valores%5B0%5D%0A%20%20%20%20for%20i%20in%20range%281%2C%20len%28valores%29%29%3A%0A%20%20%20%20%20%20%20%20if%20valores%5Bi%5D%20%3E%20mayor%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20mayor%20%3D%20valores%5Bi%5D%0A%20%20%20%20return%20mayor%0A%0Adef%20buscar_menor%28valores%29%3A%0A%20%20%20%20menor%20%3D%20valores%5B0%5D%0A%20%20%20%20for%20i%20in%20range%281%2C%20len%28valores%29%29%3A%0A%20%20%20%20%20%20%20%20if%20valores%5Bi%5D%20%3C%20menor%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20menor%20%3D%20valores%5Bi%5D%0A%20%20%20%20return%20menor%0A%0Anombres%20%3D%20%5B%22Ana%22%2C%20%22Beto%22%2C%20%22Carla%22%2C%20%22Dario%22%5D%0Anotas%20%3D%20%5B%5B3.0%2C%203.5%2C%202.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.5%2C%204.0%2C%205.0%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B2.0%2C%203.0%2C%203.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.0%2C%204.5%2C%203.0%5D%5D%0APESOS%20%3D%20%5B0.3%2C%200.3%2C%200.4%5D%0A%0Afinales%20%3D%20calcular_notas_finales%28notas%2C%20PESOS%29%0Aprint%28f%22Mas%20alta%3A%20%7Bbuscar_mayor%28finales%29%3A.2f%7D%22%29%0Aprint%28f%22Mas%20baja%3A%20%7Bbuscar_menor%28finales%29%3A.2f%7D%22%29%0A%0Aassert%20buscar_mayor%28%5B2.95%2C%204.55%2C%202.9%2C%203.75%5D%29%20%3D%3D%204.55%0Aassert%20buscar_menor%28%5B2.95%2C%204.55%2C%202.9%2C%203.75%5D%29%20%3D%3D%202.9%0Aassert%20buscar_mayor%28%5B3.5%5D%29%20%3D%3D%203.5%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%20limite%3A%20un%20elemento%0Aprint%28%22Paso%202%3A%20todas%20las%20pruebas%20pasaron%22%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=311&rawInputLstJSON=%5B%5D&textReferences=false)

---

## Parte 5 — Paso 3: los empates

`buscar_mayor` dice **cuál** es la nota más alta, pero no **de quién**, y pueden ser varios. `posiciones_iguales` retorna las posiciones de todos los que tienen ese valor. Sin `append`, sigue el patrón de `posiciones_mayores_que` de la clase 11: **contar**, **crear del tamaño justo** y **llenar** con un segundo índice `k`.

In [ ]:
def contar_iguales(valores, valor):
    """
    Supone:
        valores es una lista de numeros (puede estar vacia).
    Garantiza:
        Retorna cuantos elementos de valores son iguales a valor.
        No modifica valores.
    """
    cantidad = 0
    for i in range(len(valores)):
        if valores[i] == valor:
            cantidad = cantidad + 1
    return cantidad


def posiciones_iguales(valores, valor):
    """
    Supone:
        valores es una lista de numeros (puede estar vacia).
    Garantiza:
        Retorna una lista nueva con las posiciones de valores donde hay
        un elemento igual a valor, en orden creciente. No modifica
        valores.
    """
    cantidad = contar_iguales(valores, valor)     # 1. contar
    posiciones = [0] * cantidad                   # 2. crear del tamano exacto
    k = 0                                         # 3. llenar
    for i in range(len(valores)):
        if valores[i] == valor:
            posiciones[k] = i
            k = k + 1
    return posiciones

In [ ]:
assert contar_iguales([4.0, 3.0, 4.0], 4.0) == 2
assert posiciones_iguales([4.0, 3.0, 4.0], 4.0) == [0, 2]      # empate
assert posiciones_iguales([4.0, 3.0], 5.0) == []               # no esta
assert posiciones_iguales([], 4.0) == []                       # lista vacia
print(posiciones_iguales([4.0, 3.0, 4.0], 4.0))
print("Paso 3: todas las pruebas pasaron")

[0, 2]
Paso 3: todas las pruebas pasaron


🔎 [Ver paso a paso en Python Tutor](https://pythontutor.com/visualize.html#code=def%20contar_iguales%28valores%2C%20valor%29%3A%0A%20%20%20%20cantidad%20%3D%200%0A%20%20%20%20for%20i%20in%20range%28len%28valores%29%29%3A%0A%20%20%20%20%20%20%20%20if%20valores%5Bi%5D%20%3D%3D%20valor%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20cantidad%20%3D%20cantidad%20%2B%201%0A%20%20%20%20return%20cantidad%0A%0Adef%20posiciones_iguales%28valores%2C%20valor%29%3A%0A%20%20%20%20cantidad%20%3D%20contar_iguales%28valores%2C%20valor%29%20%20%20%20%20%23%201.%20contar%0A%20%20%20%20posiciones%20%3D%20%5B0%5D%20%2A%20cantidad%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%202.%20crear%20del%20tamano%20exacto%0A%20%20%20%20k%20%3D%200%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%203.%20llenar%0A%20%20%20%20for%20i%20in%20range%28len%28valores%29%29%3A%0A%20%20%20%20%20%20%20%20if%20valores%5Bi%5D%20%3D%3D%20valor%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20posiciones%5Bk%5D%20%3D%20i%0A%20%20%20%20%20%20%20%20%20%20%20%20k%20%3D%20k%20%2B%201%0A%20%20%20%20return%20posiciones%0A%0Aassert%20contar_iguales%28%5B4.0%2C%203.0%2C%204.0%5D%2C%204.0%29%20%3D%3D%202%0Aassert%20posiciones_iguales%28%5B4.0%2C%203.0%2C%204.0%5D%2C%204.0%29%20%3D%3D%20%5B0%2C%202%5D%20%20%20%20%20%20%23%20empate%0Aassert%20posiciones_iguales%28%5B4.0%2C%203.0%5D%2C%205.0%29%20%3D%3D%20%5B%5D%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%20no%20esta%0Aassert%20posiciones_iguales%28%5B%5D%2C%204.0%29%20%3D%3D%20%5B%5D%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%20lista%20vacia%0Aprint%28posiciones_iguales%28%5B4.0%2C%203.0%2C%204.0%5D%2C%204.0%29%29%0Aprint%28%22Paso%203%3A%20todas%20las%20pruebas%20pasaron%22%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=311&rawInputLstJSON=%5B%5D&textReferences=false)

Con los datos de la clase no hay empate. Para verlo, se inventa un curso donde dos estudiantes tienen las mismas notas:

In [ ]:
empate = calcular_notas_finales([[4.0, 4.0, 4.0],
                                 [3.0, 3.0, 3.0],
                                 [4.0, 4.0, 4.0]], PESOS)
mayor = buscar_mayor(empate)
print(empate, "->", posiciones_iguales(empate, mayor))

[4.0, 3.0, 4.0] -> [0, 2]


🔎 [Ver paso a paso en Python Tutor](https://pythontutor.com/visualize.html#code=def%20calcular_notas_finales%28notas%2C%20pesos%29%3A%0A%20%20%20%20n%20%3D%20len%28notas%29%0A%20%20%20%20finales%20%3D%20%5B0.0%5D%20%2A%20n%0A%20%20%20%20for%20i%20in%20range%28n%29%3A%0A%20%20%20%20%20%20%20%20suma%20%3D%200.0%0A%20%20%20%20%20%20%20%20for%20j%20in%20range%28len%28pesos%29%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20suma%20%3D%20suma%20%2B%20notas%5Bi%5D%5Bj%5D%20%2A%20pesos%5Bj%5D%0A%20%20%20%20%20%20%20%20finales%5Bi%5D%20%3D%20suma%0A%20%20%20%20return%20finales%0A%0Adef%20buscar_mayor%28valores%29%3A%0A%20%20%20%20mayor%20%3D%20valores%5B0%5D%0A%20%20%20%20for%20i%20in%20range%281%2C%20len%28valores%29%29%3A%0A%20%20%20%20%20%20%20%20if%20valores%5Bi%5D%20%3E%20mayor%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20mayor%20%3D%20valores%5Bi%5D%0A%20%20%20%20return%20mayor%0A%0Adef%20contar_iguales%28valores%2C%20valor%29%3A%0A%20%20%20%20cantidad%20%3D%200%0A%20%20%20%20for%20i%20in%20range%28len%28valores%29%29%3A%0A%20%20%20%20%20%20%20%20if%20valores%5Bi%5D%20%3D%3D%20valor%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20cantidad%20%3D%20cantidad%20%2B%201%0A%20%20%20%20return%20cantidad%0A%0Adef%20posiciones_iguales%28valores%2C%20valor%29%3A%0A%20%20%20%20cantidad%20%3D%20contar_iguales%28valores%2C%20valor%29%20%20%20%20%20%23%201.%20contar%0A%20%20%20%20posiciones%20%3D%20%5B0%5D%20%2A%20cantidad%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%202.%20crear%20del%20tamano%20exacto%0A%20%20%20%20k%20%3D%200%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%203.%20llenar%0A%20%20%20%20for%20i%20in%20range%28len%28valores%29%29%3A%0A%20%20%20%20%20%20%20%20if%20valores%5Bi%5D%20%3D%3D%20valor%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20posiciones%5Bk%5D%20%3D%20i%0A%20%20%20%20%20%20%20%20%20%20%20%20k%20%3D%20k%20%2B%201%0A%20%20%20%20return%20posiciones%0A%0APESOS%20%3D%20%5B0.3%2C%200.3%2C%200.4%5D%0A%0Aempate%20%3D%20calcular_notas_finales%28%5B%5B4.0%2C%204.0%2C%204.0%5D%2C%0A%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%5B3.0%2C%203.0%2C%203.0%5D%2C%0A%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%5B4.0%2C%204.0%2C%204.0%5D%5D%2C%20PESOS%29%0Amayor%20%3D%20buscar_mayor%28empate%29%0Aprint%28empate%2C%20%22-%3E%22%2C%20posiciones_iguales%28empate%2C%20mayor%29%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=311&rawInputLstJSON=%5B%5D&textReferences=false)

Por último, `mostrar_estudiantes` imprime el título y el nombre de cada estudiante con esa nota:

In [ ]:
def mostrar_estudiantes(titulo, nombres, finales, valor):
    """
    Supone:
        nombres y finales tienen el mismo tamano.
    Garantiza:
        Muestra el titulo y el nombre y la nota final de cada
        estudiante cuya nota final es igual a valor (si hay empate,
        se muestran todos).
    """
    print(f"{titulo}: {valor:.2f}")
    posiciones = posiciones_iguales(finales, valor)
    for k in range(len(posiciones)):
        print(f"  {nombres[posiciones[k]]}")

---

## Parte 6 — Integración: el programa completo

El `main.py` real lee los nombres y las notas con `input()`. Aquí se usan los datos fijos para que el notebook se pueda ejecutar de una vez; el resto es igual.

In [ ]:
def main():
    finales = calcular_notas_finales(notas, PESOS)
    for i in range(len(nombres)):
        print(f"{nombres[i]}: {finales[i]:.2f}")
    mostrar_estudiantes("Nota final mas alta", nombres, finales, buscar_mayor(finales))
    mostrar_estudiantes("Nota final mas baja", nombres, finales, buscar_menor(finales))


main()

Ana: 2.95
Beto: 4.55
Carla: 2.90
Dario: 3.75
Nota final mas alta: 4.55
  Beto
Nota final mas baja: 2.90
  Carla


🔎 [Ver paso a paso en Python Tutor](https://pythontutor.com/visualize.html#code=def%20calcular_notas_finales%28notas%2C%20pesos%29%3A%0A%20%20%20%20n%20%3D%20len%28notas%29%0A%20%20%20%20finales%20%3D%20%5B0.0%5D%20%2A%20n%0A%20%20%20%20for%20i%20in%20range%28n%29%3A%0A%20%20%20%20%20%20%20%20suma%20%3D%200.0%0A%20%20%20%20%20%20%20%20for%20j%20in%20range%28len%28pesos%29%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20suma%20%3D%20suma%20%2B%20notas%5Bi%5D%5Bj%5D%20%2A%20pesos%5Bj%5D%0A%20%20%20%20%20%20%20%20finales%5Bi%5D%20%3D%20suma%0A%20%20%20%20return%20finales%0A%0Adef%20buscar_mayor%28valores%29%3A%0A%20%20%20%20mayor%20%3D%20valores%5B0%5D%0A%20%20%20%20for%20i%20in%20range%281%2C%20len%28valores%29%29%3A%0A%20%20%20%20%20%20%20%20if%20valores%5Bi%5D%20%3E%20mayor%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20mayor%20%3D%20valores%5Bi%5D%0A%20%20%20%20return%20mayor%0A%0Adef%20buscar_menor%28valores%29%3A%0A%20%20%20%20menor%20%3D%20valores%5B0%5D%0A%20%20%20%20for%20i%20in%20range%281%2C%20len%28valores%29%29%3A%0A%20%20%20%20%20%20%20%20if%20valores%5Bi%5D%20%3C%20menor%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20menor%20%3D%20valores%5Bi%5D%0A%20%20%20%20return%20menor%0A%0Adef%20contar_iguales%28valores%2C%20valor%29%3A%0A%20%20%20%20cantidad%20%3D%200%0A%20%20%20%20for%20i%20in%20range%28len%28valores%29%29%3A%0A%20%20%20%20%20%20%20%20if%20valores%5Bi%5D%20%3D%3D%20valor%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20cantidad%20%3D%20cantidad%20%2B%201%0A%20%20%20%20return%20cantidad%0A%0Adef%20posiciones_iguales%28valores%2C%20valor%29%3A%0A%20%20%20%20cantidad%20%3D%20contar_iguales%28valores%2C%20valor%29%20%20%20%20%20%23%201.%20contar%0A%20%20%20%20posiciones%20%3D%20%5B0%5D%20%2A%20cantidad%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%202.%20crear%20del%20tamano%20exacto%0A%20%20%20%20k%20%3D%200%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%203.%20llenar%0A%20%20%20%20for%20i%20in%20range%28len%28valores%29%29%3A%0A%20%20%20%20%20%20%20%20if%20valores%5Bi%5D%20%3D%3D%20valor%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20posiciones%5Bk%5D%20%3D%20i%0A%20%20%20%20%20%20%20%20%20%20%20%20k%20%3D%20k%20%2B%201%0A%20%20%20%20return%20posiciones%0A%0Adef%20mostrar_estudiantes%28titulo%2C%20nombres%2C%20finales%2C%20valor%29%3A%0A%20%20%20%20print%28f%22%7Btitulo%7D%3A%20%7Bvalor%3A.2f%7D%22%29%0A%20%20%20%20posiciones%20%3D%20posiciones_iguales%28finales%2C%20valor%29%0A%20%20%20%20for%20k%20in%20range%28len%28posiciones%29%29%3A%0A%20%20%20%20%20%20%20%20print%28f%22%20%20%7Bnombres%5Bposiciones%5Bk%5D%5D%7D%22%29%0A%0Anombres%20%3D%20%5B%22Ana%22%2C%20%22Beto%22%2C%20%22Carla%22%2C%20%22Dario%22%5D%0Anotas%20%3D%20%5B%5B3.0%2C%203.5%2C%202.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.5%2C%204.0%2C%205.0%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B2.0%2C%203.0%2C%203.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.0%2C%204.5%2C%203.0%5D%5D%0APESOS%20%3D%20%5B0.3%2C%200.3%2C%200.4%5D%0A%0Adef%20main%28%29%3A%0A%20%20%20%20finales%20%3D%20calcular_notas_finales%28notas%2C%20PESOS%29%0A%20%20%20%20for%20i%20in%20range%28len%28nombres%29%29%3A%0A%20%20%20%20%20%20%20%20print%28f%22%7Bnombres%5Bi%5D%7D%3A%20%7Bfinales%5Bi%5D%3A.2f%7D%22%29%0A%20%20%20%20mostrar_estudiantes%28%22Nota%20final%20mas%20alta%22%2C%20nombres%2C%20finales%2C%20buscar_mayor%28finales%29%29%0A%20%20%20%20mostrar_estudiantes%28%22Nota%20final%20mas%20baja%22%2C%20nombres%2C%20finales%2C%20buscar_menor%28finales%29%29%0A%0A%0Amain%28%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=311&rawInputLstJSON=%5B%5D&textReferences=false)

El resultado coincide con el caso calculado a mano. En [`notas_matriz/`](https://github.com/algoritmos-UdeA/logica1-20262/tree/main/clases/14/teoria/codigo/notas_matriz) el mismo programa está en tres archivos: `notas.py` (las funciones), `test_notas.py` (las pruebas) y `main.py` (el programa con `input()`).

---

## Parte 7 — Errores típicos

> ⚠️ **Error frecuente.** Escribir los pesos en otro orden. Python no lo nota: solo cambian los resultados.

In [ ]:
PESOS_MAL = [0.4, 0.3, 0.3]        # el parcial 1 con el peso del 3
finales = calcular_notas_finales(notas, PESOS_MAL)
print([round(f, 2) for f in finales])

[3.0, 4.5, 2.75, 3.85]


🔎 [Ver paso a paso en Python Tutor](https://pythontutor.com/visualize.html#code=def%20calcular_notas_finales%28notas%2C%20pesos%29%3A%0A%20%20%20%20n%20%3D%20len%28notas%29%0A%20%20%20%20finales%20%3D%20%5B0.0%5D%20%2A%20n%0A%20%20%20%20for%20i%20in%20range%28n%29%3A%0A%20%20%20%20%20%20%20%20suma%20%3D%200.0%0A%20%20%20%20%20%20%20%20for%20j%20in%20range%28len%28pesos%29%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20suma%20%3D%20suma%20%2B%20notas%5Bi%5D%5Bj%5D%20%2A%20pesos%5Bj%5D%0A%20%20%20%20%20%20%20%20finales%5Bi%5D%20%3D%20suma%0A%20%20%20%20return%20finales%0A%0Anombres%20%3D%20%5B%22Ana%22%2C%20%22Beto%22%2C%20%22Carla%22%2C%20%22Dario%22%5D%0Anotas%20%3D%20%5B%5B3.0%2C%203.5%2C%202.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.5%2C%204.0%2C%205.0%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B2.0%2C%203.0%2C%203.5%5D%2C%0A%20%20%20%20%20%20%20%20%20%5B4.0%2C%204.5%2C%203.0%5D%5D%0APESOS%20%3D%20%5B0.3%2C%200.3%2C%200.4%5D%0A%0APESOS_MAL%20%3D%20%5B0.4%2C%200.3%2C%200.3%5D%20%20%20%20%20%20%20%20%23%20el%20parcial%201%20con%20el%20peso%20del%203%0Afinales%20%3D%20calcular_notas_finales%28notas%2C%20PESOS_MAL%29%0Aprint%28%5Bround%28f%2C%202%29%20for%20f%20in%20finales%5D%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=311&rawInputLstJSON=%5B%5D&textReferences=false)

Con estos pesos la más baja sería Carla igual, pero Ana pasaría de 2.95 a 3.00. Ningún mensaje lo advierte: por eso las pruebas comparan con el caso calculado a mano.

> ⚠️ **Error frecuente.** Comparar números reales con `==` cuando salen de cuentas distintas.

In [ ]:
print(0.1 + 0.2 == 0.3)
print(round(0.1 + 0.2, 2) == 0.3)

False
True


🔎 [Ver paso a paso en Python Tutor](https://pythontutor.com/visualize.html#code=print%280.1%20%2B%200.2%20%3D%3D%200.3%29%0Aprint%28round%280.1%20%2B%200.2%2C%202%29%20%3D%3D%200.3%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=311&rawInputLstJSON=%5B%5D&textReferences=false)

En `posiciones_iguales` sí se puede usar `==`, porque `mayor` **es uno de** los valores del arreglo, calculado de la misma forma.

---

## Parte 8 — Para practicar

### ✏️ Ejercicio 1 — `contar_aprobados`

Escriba una función `contar_aprobados(finales, nota_minima)` que retorne cuántos estudiantes tienen una nota final mayor o igual a `nota_minima`. Con los datos de la clase y 3.0 debe dar 2.

<details>
<summary>Ver solución</summary>

```python
def contar_aprobados(finales, nota_minima):
    aprobados = 0
    for i in range(len(finales)):
        if finales[i] >= nota_minima:
            aprobados = aprobados + 1
    return aprobados

finales = calcular_notas_finales(notas, PESOS)
assert contar_aprobados(finales, 3.0) == 2
assert contar_aprobados(finales, 5.0) == 0
```

</details>

### ✏️ Ejercicio 2 — El parcial más difícil

Escriba una función `parcial_mas_dificil(notas)` que retorne la posición de la **columna** con el promedio más bajo. Con los datos de la clase debe dar 0 (el parcial 1, promedio 3.375).

<details>
<summary>Ver solución</summary>

Primero el promedio de cada columna (un acumulador por columna) y después la posición del menor:

```python
def parcial_mas_dificil(notas):
    n = len(notas)
    m = len(notas[0])
    pos = 0
    menor = 0.0
    for j in range(m):
        suma = 0.0
        for i in range(n):
            suma = suma + notas[i][j]
        promedio = suma / n
        if j == 0 or promedio < menor:
            menor = promedio
            pos = j
    return pos

assert parcial_mas_dificil(notas) == 0
assert parcial_mas_dificil([[5.0, 1.0]]) == 1
```

</details>

---

## Parte 9 — Referencias

* [Clase 14](https://github.com/algoritmos-UdeA/logica1-20262/blob/main/clases/14/README.md) y su notebook [Arreglos bidimensionales (matrices)](https://github.com/algoritmos-UdeA/logica1-20262/blob/main/clases/14/teoria/notebooks/matrices.ipynb).
* [Clase 11](https://github.com/algoritmos-UdeA/logica1-20262/blob/main/clases/11/README.md): `posiciones_mayores_que`, el patrón de los empates.
* [Clase 15](https://github.com/algoritmos-UdeA/logica1-20262/blob/main/clases/15/README.md): el mismo programa, con los datos leídos de un archivo.
* Mera Banguero, C. A. Arreglos bidimensionales: el ejercicio de las notas parciales en que se basa este problema.
* Documentación de Python: [Aritmética de punto flotante](https://docs.python.org/es/3/tutorial/floatingpoint.html), por qué `0.1 + 0.2 != 0.3`.